# IBS with Volatility Thresholds

Combines Internal Bar Strength signals with dynamic volatility thresholds.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
================================================================================
THRESHOLD REGIME TRADING STRATEGY - VERSION 2.4 WITH IBS TIMING (CORRECTED)
================================================================================

CRITICAL FIXES FROM V2.3:
-------------------------
9. HYBRID P&L DOUBLE-COUNTING BUG FIXED (V2.4):
   - OLD (buggy): Execution days used Open[T]/Open[T-1], which overlapped with
     previous holding day's Close[T-2]→Close[T-1] in the segment Open[T-1]→Close[T-1]
   - NEW (correct): Execution days split into non-overlapping segments:
     a) OLD position: overnight = Open[T]/Close[T-1] - 1 (REALIZED - added to cash)
     b) NEW position: intraday = Close[T]/Open[T] - 1 (UNREALIZED - in position_value)

10. CASH ACCOUNTING FIX (V2.4):
    - OLD (buggy): Holding day added gross_return to cash, double-counting P&L
      (once in cash, once in position_value = shares * close_price)
    - NEW (correct): Cash only changes on actual transactions:
      * trade_cash_flow (buying/selling shares)
      * transaction_cost (commissions, spread, impact)
      * margin_interest / borrow_fee (short costs)
      * pnl_overnight (realized gain from closing old position)
    - Unrealized P&L captured ONLY in position_value, not in cash.

PREVIOUS FIXES (V2.3):
----------------------
1. LEVERAGE BUG FIXED: Signals are now +/-1 (direction only), leverage applied
   ONLY in backtest sizing. No double multiplication.

2. SIZING WITH CLOSE[T-1]: Position sizing uses Close[T-1] price, execution
   happens at Open[T]. This matches the trading flow specification.

3. COMPOUNDING MODE FIXED: When True, sizes on full mark-to-market equity
   (position_value + cash), not initial_capital + cash_reserve.

4. BENCHMARK FIXED: Uses downloaded URTH returns for relative metrics,
   not the asset's own returns.

5. PERSISTENCE FILTER: Removed forward-looking loop. Now purely retrospective
   counting of consecutive days in same regime.

6. ROBUSTNESS GUARDS: Deployment requires GRI>0.65 AND PSR>=0.95 AND WFE>=0.5.
   Rolling IC added to OOS validation.

7. CAPITAL FIX: Data filtered BEFORE backtest so portfolio starts at
   initial_capital on first day of backtest period.

TRADING FLOW (Hybrid Returns V2 - NO OVERLAP):
----------------------------------------------
Signal at Close[T-1] -> Size with Close[T-1] -> Execute at Open[T]

Day T P&L:
- If HOLDING (no signal change):
    gross_return = shares * Close[T-1] * (Close[T]/Close[T-1] - 1)
    cash: UNCHANGED (unrealized gain in position_value only)

- If EXECUTION (signal changed):
    pnl_overnight = shares_old * Close[T-1] * (Open[T]/Close[T-1] - 1)  [REALIZED]
    pnl_intraday = shares_new * Open[T] * (Close[T]/Open[T] - 1)        [UNREALIZED]
    cash += trade_cash_flow + pnl_overnight - costs

CASH ACCOUNTING:
----------------
cash changes ONLY when money actually moves:
  + trade_cash_flow: (old_shares - new_shares) * open_price
  + pnl_overnight: realized from closing old position
  - transaction_cost: commissions + spread + impact
  - margin_interest: for short positions
  - borrow_fee: for short positions

portfolio_value = cash + position_value (no double-counting)

================================================================================
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import warnings
from scipy import stats
from itertools import product
import json

warnings.filterwarnings("ignore")
TICKER = "GOOGL"


@dataclass
class StrategyConfig:
    """
    Master configuration for Threshold Regime Strategy V2.3.

    CRITICAL: Signals are ±1 (direction only). Leverage applied in backtest.
    """

    ticker: str = TICKER
    benchmark_ticker: str = "URTH"
    start_date: str = "2013-01-01"
    end_date: str = "2018-12-31"
    warmup_buffer_days: int = 700
    vol_lookback: int = 20
    quantile_window: int = 252
    n_regimes: int = 2
    regime_quantile: float = 0.5
    yz_k_numerator: float = 0.34
    yz_k_denominator_constant: float = 1.34
    yz_annualization_factor: int = 252
    persistence_days: int = 3
    use_persistence_filter: bool = True
    use_ibs_filter: bool = True
    ibs_oversold_threshold: float = 0.2
    ibs_overbought_threshold: float = 0.8
    ibs_exit_long_threshold: float = 0.85
    ibs_exit_short_threshold: float = 0.15
    use_ibs_exit: bool = True
    ibs_smoothing: int = 0
    max_wait_days: int = 5
    initial_capital: float = 100000.0
    compounding_mode: bool = True
    leverage: float = 1.0
    low_vol_allocation: float = 1.0
    high_vol_allocation: float = -1.0
    commission_per_share: float = 0.0035
    commission_min: float = 0.35
    commission_max_pct: float = 0.01
    spread_bps: float = 2.0
    market_impact_bps: float = 3.0
    margin_rate_annual: float = 0.06
    short_borrow_fee_annual: float = 0.005
    min_train_window: int = 504
    validation_window: int = 126
    step_size: int = 63
    continuous_equity: bool = False
    vol_lookback_grid: List[int] = field(default_factory=lambda: [10, 15, 20, 30, 42, 63])
    quantile_window_grid: List[int] = field(default_factory=lambda: [126, 189, 252, 378, 504])
    regime_quantile_grid: List[float] = field(default_factory=lambda: [0.3, 0.4, 0.5, 0.6, 0.7])
    persistence_days_grid: List[int] = field(default_factory=lambda: [1, 2, 3, 4, 5])
    ibs_oversold_grid: List[float] = field(default_factory=lambda: [0.1, 0.15, 0.2, 0.25, 0.3])
    ibs_overbought_grid: List[float] = field(default_factory=lambda: [0.7, 0.75, 0.8, 0.85, 0.9])
    max_wait_days_grid: List[int] = field(default_factory=lambda: [0, 3, 5, 7, 10])
    fast_mode: bool = False
    optimization_objective: str = "sharpe"
    max_param_combinations: int = 200
    gri_threshold: float = 0.65
    min_psr: float = 0.95
    min_wfe: float = 0.5
    num_trials: int = 100
    psr_benchmark_sharpe: float = 0.0


class FMPDataFetcher:
    """Yahoo Finance data fetcher."""

    def __init__(self):
        pass

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        print(f"[DATA] Fetching {ticker} from {start_date} to {end_date}...")
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
        except RuntimeError as e:
            raise ValueError(f"API request failed for {ticker}: {str(e)}")
        if "historical" not in data or len(data["historical"]) == 0:
            raise ValueError(f"No historical data for {ticker}")
        df = pd.DataFrame(data["historical"])
        df["date"] = pd.to_datetime(df["date"])
        df.set_index("date", inplace=True)
        df.sort_index(inplace=True)
        col_map = {
            "open": "Open",
            "high": "High",
            "low": "Low",
            "close": "Close",
            "adjClose": "Adj Close",
            "volume": "Volume",
        }
        available = {k: v for k, v in col_map.items() if k in df.columns}
        df = df[list(available.keys())].rename(columns=available)
        if "Adj Close" in df.columns and "Close" in df.columns:
            adj_ratio = df["Adj Close"] / df["Close"]
            df["Open"] = df["Open"] * adj_ratio
            df["High"] = df["High"] * adj_ratio
            df["Low"] = df["Low"] * adj_ratio
            df["Close"] = df["Adj Close"]
        print(f"[DATA] Retrieved {len(df)} trading days for {ticker}")
        return df


class YangZhangVolatility:

    @staticmethod
    def calculate(df: pd.DataFrame, window: int, config: StrategyConfig) -> pd.Series:
        log_open = np.log(df["Open"])
        log_high = np.log(df["High"])
        log_low = np.log(df["Low"])
        log_close = np.log(df["Close"])
        log_close_prev = log_close.shift(1)
        overnight_ret = log_open - log_close_prev
        var_overnight = overnight_ret.rolling(window).var()
        open_close_ret = log_close - log_open
        var_open_close = open_close_ret.rolling(window).var()
        rs_component = (log_high - log_close) * (log_high - log_open) + (log_low - log_close) * (
            log_low - log_open
        )
        var_rs = rs_component.rolling(window).mean()
        k = config.yz_k_numerator / (config.yz_k_denominator_constant + (window + 1) / (window - 1))
        yz_variance = var_overnight + k * var_open_close + (1 - k) * var_rs
        return np.sqrt(yz_variance) * np.sqrt(config.yz_annualization_factor)


class IBSCalculator:

    @staticmethod
    def calculate(df: pd.DataFrame, smoothing: int = 0) -> pd.Series:
        high_low_range = df["High"] - df["Low"]
        ibs = np.where(high_low_range > 0, (df["Close"] - df["Low"]) / high_low_range, 0.5)
        ibs_series = pd.Series(ibs, index=df.index, name="IBS")
        if smoothing > 0:
            ibs_series = ibs_series.rolling(smoothing).mean()
        return ibs_series


class ThresholdRegimeDetectorWithIBS:

    def __init__(
        self,
        config: StrategyConfig,
        vol_lookback: Optional[int] = None,
        quantile_window: Optional[int] = None,
        regime_quantile: Optional[float] = None,
        persistence_days: Optional[int] = None,
        ibs_oversold: Optional[float] = None,
        ibs_overbought: Optional[float] = None,
        max_wait_days: Optional[int] = None,
    ):
        self.config = config
        self.vol_lookback = vol_lookback or config.vol_lookback
        self.quantile_window = quantile_window or config.quantile_window
        self.regime_quantile = regime_quantile or config.regime_quantile
        self.persistence_days = persistence_days or config.persistence_days
        self.ibs_oversold = ibs_oversold or config.ibs_oversold_threshold
        self.ibs_overbought = ibs_overbought or config.ibs_overbought_threshold
        self.max_wait_days = max_wait_days if max_wait_days is not None else config.max_wait_days

    def detect_regimes(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["volatility"] = YangZhangVolatility.calculate(df, self.vol_lookback, self.config)
        df["vol_threshold"] = (
            df["volatility"]
            .rolling(self.quantile_window, closed="left")
            .quantile(self.regime_quantile)
        )
        df["regime_raw"] = (df["volatility"] > df["vol_threshold"]).astype(int)
        if self.config.use_persistence_filter and self.persistence_days > 1:
            df["regime"] = self._apply_persistence_filter_retrospective(
                df["regime_raw"], self.persistence_days
            )
        else:
            df["regime"] = df["regime_raw"]
        df["regime_name"] = df["regime"].map({0: "Low Vol (LONG)", 1: "High Vol (SHORT)"})
        df["ibs"] = IBSCalculator.calculate(df, self.config.ibs_smoothing)
        df["returns"] = df["Close"].pct_change()
        return df

    def _apply_persistence_filter_retrospective(
        self, raw_regime: pd.Series, persistence_days: int
    ) -> pd.Series:
        """Purely retrospective persistence filter."""
        filtered = raw_regime.copy()
        regime_streak = pd.Series(index=raw_regime.index, dtype=int)
        prev_val = None
        streak = 0
        for idx in raw_regime.index:
            val = raw_regime.loc[idx]
            if pd.isna(val):
                regime_streak.loc[idx] = 0
                continue
            if val == prev_val:
                streak += 1
            else:
                streak = 1
                prev_val = val
            regime_streak.loc[idx] = streak
        confirmed_regime = pd.Series(index=raw_regime.index, dtype=float)
        current_confirmed = None
        for idx in raw_regime.index:
            val = raw_regime.loc[idx]
            streak = regime_streak.loc[idx]
            if pd.isna(val):
                confirmed_regime.loc[idx] = np.nan
                continue
            if current_confirmed is None:
                if streak >= persistence_days:
                    current_confirmed = val
                confirmed_regime.loc[idx] = val if current_confirmed is None else current_confirmed
            else:
                if val != current_confirmed and streak >= persistence_days:
                    current_confirmed = val
                confirmed_regime.loc[idx] = current_confirmed
        return confirmed_regime

    def generate_signals_with_ibs(self, df: pd.DataFrame) -> Tuple[pd.Series, pd.DataFrame]:
        df = self.detect_regimes(df)
        signals = pd.Series(index=df.index, dtype=float)
        signals[:] = np.nan
        diagnostics = pd.DataFrame(index=df.index)
        diagnostics["regime"] = df["regime"]
        diagnostics["ibs"] = df["ibs"]
        diagnostics["state"] = "UNKNOWN"
        diagnostics["signal"] = np.nan
        diagnostics["days_waiting"] = 0
        diagnostics["entry_reason"] = ""
        current_state = "FLAT"
        days_waiting = 0
        valid_indices = df.dropna(subset=["regime", "ibs"]).index
        if len(valid_indices) == 0:
            return (signals, diagnostics)
        for i, idx in enumerate(valid_indices):
            regime = df.loc[idx, "regime"]
            ibs = df.loc[idx, "ibs"]
            if i > 0:
                prev_regime = df.loc[valid_indices[i - 1], "regime"]
                regime_changed = regime != prev_regime and (not pd.isna(prev_regime))
            else:
                prev_regime = np.nan
                regime_changed = False
            if regime == 0:
                target_direction = "LONG"
                target_alloc = self.config.low_vol_allocation
            else:
                target_direction = "SHORT"
                target_alloc = self.config.high_vol_allocation
            entry_reason = ""
            if not self.config.use_ibs_filter:
                signals.loc[idx] = target_alloc
                current_state = target_direction
                entry_reason = "direct_regime"
            else:
                ibs_favorable_long = ibs < self.ibs_oversold
                ibs_favorable_short = ibs > self.ibs_overbought
                ibs_exit_long = ibs > self.config.ibs_exit_long_threshold
                ibs_exit_short = ibs < self.config.ibs_exit_short_threshold
                if current_state == "FLAT":
                    if target_direction == "LONG":
                        if ibs_favorable_long:
                            current_state = "LONG"
                            signals.loc[idx] = target_alloc
                            entry_reason = "ibs_oversold"
                            days_waiting = 0
                        else:
                            current_state = "WAITING_LONG"
                            signals.loc[idx] = 0.0
                            days_waiting = 1
                            entry_reason = "waiting_for_ibs"
                    elif ibs_favorable_short:
                        current_state = "SHORT"
                        signals.loc[idx] = target_alloc
                        entry_reason = "ibs_overbought"
                        days_waiting = 0
                    else:
                        current_state = "WAITING_SHORT"
                        signals.loc[idx] = 0.0
                        days_waiting = 1
                        entry_reason = "waiting_for_ibs"
                elif current_state == "WAITING_LONG":
                    if regime_changed and target_direction == "SHORT":
                        if ibs_favorable_short:
                            current_state = "SHORT"
                            signals.loc[idx] = target_alloc
                            entry_reason = "regime_flip_short"
                        else:
                            current_state = "WAITING_SHORT"
                            signals.loc[idx] = 0.0
                            entry_reason = "regime_flip_waiting_short"
                        days_waiting = 0
                    elif ibs_favorable_long:
                        current_state = "LONG"
                        signals.loc[idx] = target_alloc
                        entry_reason = f"ibs_oversold_after_{days_waiting}d"
                        days_waiting = 0
                    elif self.max_wait_days > 0 and days_waiting >= self.max_wait_days:
                        current_state = "LONG"
                        signals.loc[idx] = target_alloc
                        entry_reason = f"forced_after_{days_waiting}d"
                        days_waiting = 0
                    else:
                        signals.loc[idx] = 0.0
                        days_waiting += 1
                        entry_reason = f"waiting_{days_waiting}d"
                elif current_state == "WAITING_SHORT":
                    if regime_changed and target_direction == "LONG":
                        if ibs_favorable_long:
                            current_state = "LONG"
                            signals.loc[idx] = target_alloc
                            entry_reason = "regime_flip_long"
                        else:
                            current_state = "WAITING_LONG"
                            signals.loc[idx] = 0.0
                            entry_reason = "regime_flip_waiting_long"
                        days_waiting = 0
                    elif ibs_favorable_short:
                        current_state = "SHORT"
                        signals.loc[idx] = target_alloc
                        entry_reason = f"ibs_overbought_after_{days_waiting}d"
                        days_waiting = 0
                    elif self.max_wait_days > 0 and days_waiting >= self.max_wait_days:
                        current_state = "SHORT"
                        signals.loc[idx] = target_alloc
                        entry_reason = f"forced_after_{days_waiting}d"
                        days_waiting = 0
                    else:
                        signals.loc[idx] = 0.0
                        days_waiting += 1
                        entry_reason = f"waiting_{days_waiting}d"
                elif current_state == "LONG":
                    if regime_changed and target_direction == "SHORT":
                        if ibs_favorable_short:
                            current_state = "SHORT"
                            signals.loc[idx] = target_alloc
                            entry_reason = "regime_flip_to_short"
                        else:
                            current_state = "WAITING_SHORT"
                            signals.loc[idx] = 0.0
                            days_waiting = 1
                            entry_reason = "regime_flip_waiting_short"
                    elif self.config.use_ibs_exit and ibs_exit_long:
                        current_state = "FLAT"
                        signals.loc[idx] = 0.0
                        entry_reason = "ibs_exit_long"
                    else:
                        signals.loc[idx] = target_alloc
                        entry_reason = "holding_long"
                elif current_state == "SHORT":
                    if regime_changed and target_direction == "LONG":
                        if ibs_favorable_long:
                            current_state = "LONG"
                            signals.loc[idx] = target_alloc
                            entry_reason = "regime_flip_to_long"
                        else:
                            current_state = "WAITING_LONG"
                            signals.loc[idx] = 0.0
                            days_waiting = 1
                            entry_reason = "regime_flip_waiting_long"
                    elif self.config.use_ibs_exit and ibs_exit_short:
                        current_state = "FLAT"
                        signals.loc[idx] = 0.0
                        entry_reason = "ibs_exit_short"
                    else:
                        signals.loc[idx] = target_alloc
                        entry_reason = "holding_short"
            diagnostics.loc[idx, "state"] = current_state
            diagnostics.loc[idx, "signal"] = signals.loc[idx]
            diagnostics.loc[idx, "days_waiting"] = days_waiting
            diagnostics.loc[idx, "entry_reason"] = entry_reason
        return (signals, diagnostics)


class TransactionCostCalculator:

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_total_cost(self, shares: float, price: float) -> float:
        shares = abs(shares)
        trade_value = shares * price
        raw_comm = shares * self.config.commission_per_share
        commission = max(
            self.config.commission_min, min(raw_comm, trade_value * self.config.commission_max_pct)
        )
        spread = trade_value * (self.config.spread_bps / 10000)
        impact = trade_value * (self.config.market_impact_bps / 10000)
        return commission + spread + impact

    def calculate_margin_interest(self, borrowed_value: float, days: int = 1) -> float:
        daily_rate = self.config.margin_rate_annual / 252
        return abs(borrowed_value) * daily_rate * days

    def calculate_short_borrow_fee(self, short_value: float, days: int = 1) -> float:
        daily_rate = self.config.short_borrow_fee_annual / 252
        return abs(short_value) * daily_rate * days


class BacktestEngine:
    """
    Production-grade backtesting engine.

    FIXES APPLIED:
    1. Leverage applied ONCE in sizing
    2. Sizing uses Close[T-1], execution at Open[T]
    3. Hybrid P&L V2 (NO DOUBLE-COUNTING):
       - Holding days: Close[T]/Close[T-1] - 1 (UNREALIZED - cash unchanged)
       - Execution days: Split into two non-overlapping segments:
         a) OLD position: overnight return = Open[T]/Close[T-1] - 1 (REALIZED)
         b) NEW position: intraday return = Close[T]/Open[T] - 1 (UNREALIZED)
    4. Compounding uses full mark-to-market equity
    5. CASH ACCOUNTING: Cash only changes on actual transactions:
       - trade_cash_flow (buy/sell)
       - pnl_overnight (realized from closing old position)
       - transaction_cost, margin_interest, borrow_fee

    TIMELINE (NO OVERLAP, NO DOUBLE-COUNTING):
    ------------------------------------------
    Day T-1 (Holding):     Close[T-2] ════════════════════► Close[T-1]
                           cash: unchanged (unrealized in position_value)
                                                            │
    Day T (Execution):                                      │
      - Old position:                              Close[T-1]──►Open[T]  (overnight, REALIZED → cash)
      - New position:                                          Open[T]══►Close[T] (intraday, UNREALIZED)

    portfolio_value = cash + (shares * close_price)  [no double-counting]
    """

    def __init__(self, config: StrategyConfig, benchmark_returns: Optional[pd.Series] = None):
        self.config = config
        self.cost_calculator = TransactionCostCalculator(config)
        self.benchmark_returns = benchmark_returns

    def run_backtest(self, df: pd.DataFrame, signals: pd.Series) -> pd.DataFrame:
        df = df.copy()
        df["target_allocation"] = signals.shift(1)
        df["close_prev"] = df["Close"].shift(1)
        df["open_prev"] = df["Open"].shift(1)
        results = {
            "date": [],
            "open_price": [],
            "close_price": [],
            "target_alloc": [],
            "shares": [],
            "position_value": [],
            "cash": [],
            "portfolio_value": [],
            "daily_return": [],
            "gross_return": [],
            "transaction_cost": [],
            "margin_interest": [],
            "borrow_fee": [],
            "net_return": [],
            "benchmark_return": [],
            "is_execution_day": [],
            "regime": [],
            "position_type": [],
            "ibs": [],
            "sizing_price": [],
            "pnl_overnight": [],
            "pnl_intraday": [],
        }
        cash = self.config.initial_capital
        shares = 0.0
        executed_signal = None
        valid_dates = df.dropna(subset=["target_allocation", "Open", "Close", "close_prev"]).index
        if len(valid_dates) == 0:
            raise ValueError("No valid trading dates")
        for i, date in enumerate(valid_dates):
            open_price = df.loc[date, "Open"]
            close_price = df.loc[date, "Close"]
            close_prev = df.loc[date, "close_prev"]
            open_prev = df.loc[date, "open_prev"]
            target_alloc = df.loc[date, "target_allocation"]
            regime = df.loc[date, "regime"] if "regime" in df.columns else np.nan
            ibs = df.loc[date, "ibs"] if "ibs" in df.columns else np.nan
            signal_changed = executed_signal is None or abs(target_alloc - executed_signal) > 0.01
            is_execution_day = signal_changed
            transaction_cost = 0.0
            margin_interest = 0.0
            borrow_fee = 0.0
            gross_return = 0.0
            pnl_overnight = 0.0
            pnl_intraday = 0.0
            if signal_changed:
                if self.config.compounding_mode:
                    position_value_prev = shares * close_prev
                    sizing_capital = cash + position_value_prev
                else:
                    sizing_capital = self.config.initial_capital
                target_value = target_alloc * sizing_capital * self.config.leverage
                new_shares = target_value / close_prev if close_prev > 0 else 0
                shares_change = new_shares - shares
                if abs(shares_change) > 0.001:
                    transaction_cost = self.cost_calculator.calculate_total_cost(
                        shares_change, open_price
                    )
                if shares != 0 and close_prev > 0:
                    overnight_return = open_price / close_prev - 1
                    pnl_overnight = shares * close_prev * overnight_return
                else:
                    pnl_overnight = 0.0
                old_shares = shares
                shares = new_shares
                executed_signal = target_alloc
                if shares != 0 and open_price > 0:
                    intraday_return = close_price / open_price - 1
                    pnl_intraday = shares * open_price * intraday_return
                else:
                    pnl_intraday = 0.0
                gross_return = pnl_overnight + pnl_intraday
                trade_cash_flow = (old_shares - new_shares) * open_price
                cash = cash + trade_cash_flow - transaction_cost + pnl_overnight
            elif shares != 0 and close_prev > 0:
                close_return = close_price / close_prev - 1
                gross_return = shares * close_prev * close_return
            else:
                gross_return = 0.0
            if shares < 0:
                short_value = abs(shares * close_price)
                margin_interest = self.cost_calculator.calculate_margin_interest(short_value)
                borrow_fee = self.cost_calculator.calculate_short_borrow_fee(short_value)
                cash = cash - margin_interest - borrow_fee
            position_value = shares * close_price
            portfolio_value = cash + position_value
            net_return = gross_return - transaction_cost - margin_interest - borrow_fee
            if i > 0 and results["portfolio_value"]:
                prev_pv = results["portfolio_value"][-1]
                daily_return = portfolio_value / prev_pv - 1 if prev_pv > 0 else 0
            else:
                daily_return = 0.0
            if self.benchmark_returns is not None and date in self.benchmark_returns.index:
                benchmark_return = self.benchmark_returns.loc[date]
            else:
                benchmark_return = close_price / close_prev - 1 if close_prev > 0 else 0
            if shares > 0.001:
                position_type = "LONG"
            elif shares < -0.001:
                position_type = "SHORT"
            else:
                position_type = "FLAT"
            results["date"].append(date)
            results["open_price"].append(open_price)
            results["close_price"].append(close_price)
            results["target_alloc"].append(target_alloc)
            results["shares"].append(shares)
            results["position_value"].append(position_value)
            results["cash"].append(cash)
            results["portfolio_value"].append(portfolio_value)
            results["daily_return"].append(daily_return)
            results["gross_return"].append(gross_return)
            results["transaction_cost"].append(transaction_cost)
            results["margin_interest"].append(margin_interest)
            results["borrow_fee"].append(borrow_fee)
            results["net_return"].append(net_return)
            results["benchmark_return"].append(benchmark_return)
            results["is_execution_day"].append(is_execution_day)
            results["regime"].append(regime)
            results["position_type"].append(position_type)
            results["ibs"].append(ibs)
            results["sizing_price"].append(close_prev)
            results["pnl_overnight"].append(pnl_overnight)
            results["pnl_intraday"].append(pnl_intraday)
        results_df = pd.DataFrame(results)
        results_df.set_index("date", inplace=True)
        results_df["cumulative_return"] = (
            results_df["portfolio_value"] / self.config.initial_capital - 1
        )
        results_df["cumulative_benchmark"] = (1 + results_df["benchmark_return"]).cumprod() - 1
        results_df["strategy_return"] = results_df["daily_return"]
        return results_df


class PerformanceAnalytics:

    def __init__(self, config: StrategyConfig):
        self.config = config

    def calculate_basic_metrics(
        self,
        returns: pd.Series,
        benchmark_returns: Optional[pd.Series] = None,
        annualization_factor: int = 252,
    ) -> Dict[str, float]:
        returns = returns.dropna()
        if len(returns) < 20:
            return {
                "error": "Insufficient data",
                "sharpe_ratio": 0,
                "num_observations": len(returns),
            }
        total_return = (1 + returns).prod() - 1
        n_years = len(returns) / annualization_factor
        annual_return = (1 + total_return) ** (1 / n_years) - 1 if n_years > 0 else 0
        volatility = returns.std() * np.sqrt(annualization_factor)
        sharpe = annual_return / volatility if volatility > 0 else 0
        downside = returns[returns < 0]
        downside_dev = downside.std() * np.sqrt(annualization_factor) if len(downside) > 0 else 0
        sortino = annual_return / downside_dev if downside_dev > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()
        calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0
        metrics = {
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "sortino_ratio": sortino,
            "max_drawdown": max_drawdown,
            "calmar_ratio": calmar,
            "skewness": returns.skew(),
            "kurtosis": returns.kurtosis() + 3,
            "num_observations": len(returns),
        }
        if benchmark_returns is not None:
            benchmark_returns = benchmark_returns.reindex(returns.index).dropna()
            if len(benchmark_returns) > 20:
                benchmark_total = (1 + benchmark_returns).prod() - 1
                active_returns = returns - benchmark_returns.reindex(returns.index).fillna(0)
                tracking_error = active_returns.std() * np.sqrt(annualization_factor)
                benchmark_annual = (1 + benchmark_total) ** (
                    1 / (len(benchmark_returns) / annualization_factor)
                ) - 1
                info_ratio = (
                    (annual_return - benchmark_annual) / tracking_error if tracking_error > 0 else 0
                )
                metrics["benchmark_return"] = benchmark_total
                metrics["excess_return"] = total_return - benchmark_total
                metrics["information_ratio"] = info_ratio
        return metrics

    def calculate_mean_ic(self, signals: pd.Series, forward_returns: pd.Series) -> float:
        aligned_signals = signals.shift(1)
        valid_mask = ~(aligned_signals.isna() | forward_returns.isna())
        if valid_mask.sum() < 20:
            return 0.0
        return aligned_signals[valid_mask].corr(forward_returns[valid_mask])

    def calculate_rolling_ic(
        self, signals: pd.Series, forward_returns: pd.Series, window: int = 21
    ) -> pd.Series:
        """
        TRUE rolling IC: correlation over moving windows within the series.

        Args:
            signals: Signal series (will be shifted by 1)
            forward_returns: Next-day returns
            window: Rolling window size (default 21 = ~1 month)

        Returns:
            Series of rolling IC values
        """
        aligned_signals = signals.shift(1)
        rolling_ic = aligned_signals.rolling(window).corr(forward_returns)
        return rolling_ic

    def calculate_rolling_ic_stats(
        self, signals: pd.Series, forward_returns: pd.Series, window: int = 21
    ) -> Dict[str, float]:
        """
        Compute statistics on rolling IC for stability assessment.

        Returns:
            Dict with mean_ic, std_ic, pct_positive, ic_ir (IC information ratio)
        """
        rolling_ic = self.calculate_rolling_ic(signals, forward_returns, window)
        valid_ic = rolling_ic.dropna()
        if len(valid_ic) < 10:
            return {"mean_ic": 0.0, "std_ic": 0.0, "pct_positive": 0.0, "ic_ir": 0.0}
        mean_ic = valid_ic.mean()
        std_ic = valid_ic.std()
        pct_positive = (valid_ic > 0).mean()
        ic_ir = mean_ic / std_ic if std_ic > 0 else 0.0
        return {"mean_ic": mean_ic, "std_ic": std_ic, "pct_positive": pct_positive, "ic_ir": ic_ir}

    def calculate_deflated_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        num_trials: int,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        if num_trials <= 1:
            return self.calculate_probabilistic_sharpe_ratio(
                sharpe, num_observations, 0, skewness, kurtosis
            )
        try:
            euler = 0.5772156649
            z_arg = max(0.0001, min(0.9999, 1 - 1 / num_trials))
            e_max_arg = max(0.0001, min(0.9999, 1 - 1 / (num_trials * np.e)))
            expected_max_sr = (1 - euler) * stats.norm.ppf(z_arg) + euler * stats.norm.ppf(
                e_max_arg
            )
            sr_std = np.sqrt(
                (1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2)
                / num_observations
            )
            expected_max_sr *= sr_std
            z_score = (sharpe - expected_max_sr) / sr_std if sr_std > 0 else 0
            return max(0, min(1, stats.norm.cdf(z_score)))
        except:
            return 0.5

    def calculate_probabilistic_sharpe_ratio(
        self,
        sharpe: float,
        num_observations: int,
        benchmark_sharpe: float = 0,
        skewness: float = 0,
        kurtosis: float = 3,
    ) -> float:
        if num_observations < 2:
            return 0.5
        variance_term = max(
            0.01, 1 + 0.5 * sharpe**2 - skewness * sharpe + (kurtosis - 3) / 4 * sharpe**2
        )
        sharpe_std = np.sqrt(variance_term / num_observations)
        z_score = (sharpe - benchmark_sharpe) / sharpe_std if sharpe_std > 0 else 0
        return max(0, min(1, stats.norm.cdf(z_score)))

    def calculate_hurst_exponent(self, returns: pd.Series, max_lag: int = 100) -> float:
        returns = returns.dropna().values
        n = len(returns)
        if n < 50:
            return 0.5
        max_lag = min(max_lag, n // 4)
        rs_values = []
        for lag in range(10, max_lag):
            num_chunks = n // lag
            rs_list = []
            for i in range(num_chunks):
                chunk = returns[i * lag : (i + 1) * lag]
                mean_c = chunk.mean()
                cum_dev = np.cumsum(chunk - mean_c)
                r = cum_dev.max() - cum_dev.min()
                s = chunk.std()
                if s > 0:
                    rs_list.append(r / s)
            if rs_list:
                rs_values.append((np.log(lag), np.log(np.mean(rs_list))))
        if len(rs_values) < 2:
            return 0.5
        rs_arr = np.array(rs_values)
        slope, _, _, _, _ = stats.linregress(rs_arr[:, 0], rs_arr[:, 1])
        return max(0, min(1, slope))

    def calculate_cdar(self, returns: pd.Series, alpha: float = 0.05) -> float:
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.cummax()
        drawdowns = (cumulative - running_max) / running_max
        threshold = drawdowns.quantile(alpha)
        worst = drawdowns[drawdowns <= threshold]
        return worst.mean() if len(worst) > 0 else drawdowns.min()

    def calculate_parameter_stability_index(self, sharpes: List[float]) -> float:
        valid = [s for s in sharpes if not np.isnan(s) and s != 0]
        if len(valid) < 2:
            return 0.5
        mean_s = np.mean(valid)
        std_s = np.std(valid)
        if mean_s != 0:
            return 1 / (1 + abs(std_s / mean_s))
        return 0.5 if std_s == 0 else 0.3

    def calculate_global_robustness_index(self, metrics: Dict[str, float]) -> float:
        weights = {"dsr": 0.2, "psr": 0.2, "wfe": 0.2, "hurst": 0.1, "cdar": 0.15, "psi": 0.15}
        normalized = {
            "dsr": max(0, min(1, metrics.get("dsr", 0.5))),
            "psr": max(0, min(1, metrics.get("psr", 0.5))),
            "wfe": max(0, min(1, metrics.get("wfe", 0.5) / 1.5)),
            "hurst": max(0, min(1, metrics.get("hurst", 0.5))),
            "cdar": max(0, min(1, 1 + metrics.get("cdar", -0.2) * 2)),
            "psi": max(0, min(1, metrics.get("psi", 0.5))),
        }
        return sum((weights[k] * normalized[k] for k in weights))


class HyperparameterOptimizer:

    def __init__(self, config: StrategyConfig, benchmark_returns: Optional[pd.Series] = None):
        self.config = config
        self.analytics = PerformanceAnalytics(config)
        self.benchmark_returns = benchmark_returns

    def generate_param_grid(self) -> List[Dict[str, Any]]:
        if self.config.fast_mode:
            vol_lb, q_win, r_q, pers = ([20, 40], [252], [0.5], [3])
            ibs_os, ibs_ob, mwd = ([0.2], [0.8], [5])
        else:
            vol_lb = self.config.vol_lookback_grid
            q_win = self.config.quantile_window_grid
            r_q = self.config.regime_quantile_grid
            pers = self.config.persistence_days_grid
            ibs_os = self.config.ibs_oversold_grid
            ibs_ob = self.config.ibs_overbought_grid
            mwd = self.config.max_wait_days_grid
        params = []
        for vl, qw, rq, pd, os, ob, wd in product(vol_lb, q_win, r_q, pers, ibs_os, ibs_ob, mwd):
            if qw >= vl and os < ob:
                params.append(
                    {
                        "vol_lookback": vl,
                        "quantile_window": qw,
                        "regime_quantile": rq,
                        "persistence_days": pd,
                        "ibs_oversold": os,
                        "ibs_overbought": ob,
                        "max_wait_days": wd,
                    }
                )
        if len(params) > self.config.max_param_combinations:
            np.random.seed(42)
            idx = np.random.choice(len(params), self.config.max_param_combinations, replace=False)
            params = [params[i] for i in idx]
        return params

    def evaluate_params(self, params: Dict, df: pd.DataFrame) -> Dict[str, float]:
        try:
            detector = ThresholdRegimeDetectorWithIBS(self.config, **params)
            signals, _ = detector.generate_signals_with_ibs(df)
            if signals.dropna().shape[0] < 60:
                return {"sharpe_ratio": -999, "valid": False}
            df_regimes = detector.detect_regimes(df)
            engine = BacktestEngine(self.config, self.benchmark_returns)
            results = engine.run_backtest(df_regimes, signals)
            returns = results["strategy_return"].dropna()
            if len(returns) < 30:
                return {"sharpe_ratio": -999, "valid": False}
            metrics = self.analytics.calculate_basic_metrics(returns, self.benchmark_returns)
            metrics["valid"] = True
            metrics["num_trades"] = results["is_execution_day"].sum()
            return metrics
        except Exception as e:
            return {"sharpe_ratio": -999, "valid": False, "error": str(e)}

    def optimize(self, df: pd.DataFrame, verbose: bool = False) -> Tuple[Dict, Dict]:
        grid = self.generate_param_grid()
        if verbose:
            print(f"    Testing {len(grid)} parameter combinations...")
        best_params, best_obj, best_metrics = (None, -np.inf, {})
        for params in grid:
            metrics = self.evaluate_params(params, df)
            if not metrics.get("valid"):
                continue
            obj = metrics.get("sharpe_ratio", -999)
            if obj > best_obj:
                best_obj, best_params, best_metrics = (obj, params, metrics)
        if best_params is None:
            best_params = {
                "vol_lookback": self.config.vol_lookback,
                "quantile_window": self.config.quantile_window,
                "regime_quantile": self.config.regime_quantile,
                "persistence_days": self.config.persistence_days,
                "ibs_oversold": self.config.ibs_oversold_threshold,
                "ibs_overbought": self.config.ibs_overbought_threshold,
                "max_wait_days": self.config.max_wait_days,
            }
        if verbose:
            print(
                f"    Best: vol_lb={best_params['vol_lookback']}, q={best_params['regime_quantile']}, Sharpe={best_obj:.3f}"
            )
        return (best_params, best_metrics)


class WalkForwardValidator:
    """
    Walk-Forward Validation with expanding window.

    MODES:
    - continuous_equity=False (default): Each fold starts fresh at initial_capital.
      Best for research/comparison - gives independent OOS Sharpe per fold.

    - continuous_equity=True: Equity chains across folds (fold N starts where N-1 ended).
      Simulates real track record. Use for pitchbooks/investor reporting.
    """

    def __init__(self, config: StrategyConfig, benchmark_returns: Optional[pd.Series] = None):
        self.config = config
        self.analytics = PerformanceAnalytics(config)
        self.optimizer = HyperparameterOptimizer(config, benchmark_returns)
        self.benchmark_returns = benchmark_returns

    def run_walk_forward(self, df: pd.DataFrame, verbose: bool = True) -> Dict[str, Any]:
        detector = ThresholdRegimeDetectorWithIBS(self.config)
        df_init = detector.detect_regimes(df)
        max_warmup = max(self.config.quantile_window_grid) + max(self.config.vol_lookback_grid)
        valid_idx = df_init.dropna(subset=["regime", "vol_threshold", "ibs"]).index
        min_req = self.config.min_train_window + self.config.validation_window + max_warmup
        if len(valid_idx) < min_req:
            raise ValueError(f"Insufficient data: need {min_req}, have {len(valid_idx)}")
        fold_results, oos_returns_list, param_history = ([], [], [])
        num_trials = len(self.optimizer.generate_param_grid())
        fold_num, train_end_idx = (0, self.config.min_train_window)
        carry_forward_capital = self.config.initial_capital
        if verbose:
            print(f"\n{'=' * 80}\nWALK-FORWARD VALIDATION - V2.4 (Hybrid P&L Fix)")
            print(
                f"Mode: {('Continuous Equity' if self.config.continuous_equity else 'Independent Folds')}"
            )
            print(f"{'=' * 80}")
        while train_end_idx + self.config.validation_window <= len(valid_idx):
            fold_num += 1
            train_idx = valid_idx[:train_end_idx]
            val_idx = valid_idx[train_end_idx : train_end_idx + self.config.validation_window]
            if verbose:
                print(
                    f"\nFold {fold_num}: Train {train_idx[0].date()}-{train_idx[-1].date()}, Val {val_idx[0].date()}-{val_idx[-1].date()}"
                )
            train_df = df.loc[train_idx]
            opt_params, is_metrics = self.optimizer.optimize(train_df, verbose=verbose)
            param_history.append({"fold": fold_num, **opt_params})
            is_sharpe = is_metrics.get("sharpe_ratio", 0)
            opt_detector = ThresholdRegimeDetectorWithIBS(self.config, **opt_params)
            full_val_df = df.loc[: val_idx[-1]]
            val_signals, _ = opt_detector.generate_signals_with_ibs(full_val_df)
            val_signals = val_signals.loc[val_idx]
            df_regimes_val = opt_detector.detect_regimes(full_val_df).loc[val_idx]
            if self.config.continuous_equity:
                fold_config = StrategyConfig()
                fold_config.initial_capital = carry_forward_capital
                engine = BacktestEngine(fold_config, self.benchmark_returns)
            else:
                engine = BacktestEngine(self.config, self.benchmark_returns)
            val_results = engine.run_backtest(df_regimes_val, val_signals)
            val_returns = val_results["strategy_return"].dropna()
            oos_returns_list.append(val_returns)
            if self.config.continuous_equity:
                carry_forward_capital = val_results["portfolio_value"].iloc[-1]
            oos_metrics = (
                self.analytics.calculate_basic_metrics(val_returns, self.benchmark_returns)
                if len(val_returns) > 10
                else {}
            )
            oos_sharpe = oos_metrics.get("sharpe_ratio", 0)
            wfe = oos_sharpe / is_sharpe if is_sharpe != 0 else 0
            forward_returns = (
                df_regimes_val["returns"]
                if "returns" in df_regimes_val.columns
                else df_regimes_val["Close"].pct_change()
            )
            ic_stats = self.analytics.calculate_rolling_ic_stats(
                val_signals, forward_returns, window=21
            )
            fold_results.append(
                {
                    "fold": fold_num,
                    "is_sharpe": is_sharpe,
                    "oos_sharpe": oos_sharpe,
                    "oos_return": oos_metrics.get("total_return", 0),
                    "wfe": wfe,
                    "mean_ic": ic_stats["mean_ic"],
                    "std_ic": ic_stats["std_ic"],
                    "pct_ic_positive": ic_stats["pct_positive"],
                    "ic_ir": ic_stats["ic_ir"],
                    "oos_long_days": (val_results["position_type"] == "LONG").sum(),
                    "oos_short_days": (val_results["position_type"] == "SHORT").sum(),
                    "oos_flat_days": (val_results["position_type"] == "FLAT").sum(),
                    "ending_capital": (
                        val_results["portfolio_value"].iloc[-1]
                        if self.config.continuous_equity
                        else self.config.initial_capital
                    ),
                    **{f"param_{k}": v for k, v in opt_params.items()},
                }
            )
            if verbose:
                print(f"  IS Sharpe: {is_sharpe:.3f}, OOS Sharpe: {oos_sharpe:.3f}, WFE: {wfe:.3f}")
                print(
                    f"  Rolling IC: mean={ic_stats['mean_ic']:.3f}, std={ic_stats['std_ic']:.3f}, %pos={ic_stats['pct_positive']:.1%}"
                )
            train_end_idx += self.config.step_size
        all_oos = pd.concat(oos_returns_list)
        agg = self.analytics.calculate_basic_metrics(all_oos, self.benchmark_returns)
        valid_wfes = [
            f["wfe"] for f in fold_results if not np.isnan(f["wfe"]) and abs(f["wfe"]) < 10
        ]
        avg_wfe = np.mean(valid_wfes) if valid_wfes else 0
        avg_ic = (
            np.mean([f["mean_ic"] for f in fold_results if not np.isnan(f["mean_ic"])])
            if fold_results
            else 0
        )
        avg_ic_ir = (
            np.mean([f["ic_ir"] for f in fold_results if not np.isnan(f["ic_ir"])])
            if fold_results
            else 0
        )
        skew, kurt = (all_oos.skew(), all_oos.kurtosis() + 3)
        dsr = self.analytics.calculate_deflated_sharpe_ratio(
            agg.get("sharpe_ratio", 0), len(all_oos), num_trials * len(fold_results), skew, kurt
        )
        psr = self.analytics.calculate_probabilistic_sharpe_ratio(
            agg.get("sharpe_ratio", 0), len(all_oos), self.config.psr_benchmark_sharpe, skew, kurt
        )
        hurst = self.analytics.calculate_hurst_exponent(all_oos)
        cdar = self.analytics.calculate_cdar(all_oos)
        psi = self.analytics.calculate_parameter_stability_index(
            [f["oos_sharpe"] for f in fold_results]
        )
        gri = self.analytics.calculate_global_robustness_index(
            {"dsr": dsr, "psr": psr, "wfe": avg_wfe, "hurst": hurst, "cdar": cdar, "psi": psi}
        )
        is_robust = (
            gri > self.config.gri_threshold
            and psr >= self.config.min_psr
            and (avg_wfe >= self.config.min_wfe)
        )
        aggregate = {
            "num_folds": len(fold_results),
            "oos_total_return": agg.get("total_return", 0),
            "oos_sharpe": agg.get("sharpe_ratio", 0),
            "oos_max_drawdown": agg.get("max_drawdown", 0),
            "avg_wfe": avg_wfe,
            "avg_ic": avg_ic,
            "avg_ic_ir": avg_ic_ir,
            "dsr": dsr,
            "psr": psr,
            "hurst": hurst,
            "cdar": cdar,
            "psi": psi,
            "gri": gri,
            "is_robust": is_robust,
            "total_long_days": sum((f["oos_long_days"] for f in fold_results)),
            "total_short_days": sum((f["oos_short_days"] for f in fold_results)),
            "total_flat_days": sum((f["oos_flat_days"] for f in fold_results)),
            "final_capital": carry_forward_capital if self.config.continuous_equity else None,
        }
        if verbose:
            print(
                f"\n{'-' * 80}\nAGGREGATE: OOS Sharpe={agg.get('sharpe_ratio', 0):.3f}, GRI={gri:.3f}, PSR={psr:.3f}, WFE={avg_wfe:.3f}"
            )
            print(f"Rolling IC: avg={avg_ic:.3f}, IC IR={avg_ic_ir:.3f}")
            if self.config.continuous_equity:
                print(
                    f"Continuous Equity: ${self.config.initial_capital:,.0f} -> ${carry_forward_capital:,.0f}"
                )
            print(f"DEPLOYMENT: {('READY' if is_robust else 'NOT READY')}")
        return {
            "fold_results": fold_results,
            "aggregate_metrics": aggregate,
            "oos_returns": all_oos,
            "param_history": param_history,
        }


def main():
    print("=" * 80)
    print(f"THRESHOLD REGIME STRATEGY V2.4 - {TICKER} (HYBRID P&L FIX APPLIED)")
    print("=" * 80)
    config = StrategyConfig()
    print(f"\n[CONFIG] {config.ticker}, {config.start_date} to {config.end_date}")
    print(f"  Capital: ${config.initial_capital:,.0f}, Leverage: {config.leverage}x")
    print(
        f"  Mode: {('Production' if config.compounding_mode else 'Research')}, IBS: {('ON' if config.use_ibs_filter else 'OFF')}"
    )
    print("\n[DATA] Fetching from Yahoo Finance...")
    start_dt = datetime.strptime(config.start_date, "%Y-%m-%d")
    ext_start = (start_dt - timedelta(days=config.warmup_buffer_days)).strftime("%Y-%m-%d")
    fetcher = FMPDataFetcher()
    try:
        df_full = fetcher.fetch_historical_data(config.ticker, ext_start, config.end_date)
    except ValueError as e:
        print(f"[ERROR] {e}")
        return None
    try:
        benchmark_df = fetcher.fetch_historical_data(
            config.benchmark_ticker, ext_start, config.end_date
        )
        benchmark_returns = benchmark_df["Close"].pct_change()
        print(f"  Benchmark {config.benchmark_ticker}: {len(benchmark_df)} days")
    except:
        print(f"  [WARNING] Could not fetch {config.benchmark_ticker}")
        benchmark_returns = None
    print("\n[WALK-FORWARD VALIDATION]")
    wf = WalkForwardValidator(config, benchmark_returns)
    wf_results = wf.run_walk_forward(df_full, verbose=True)
    print("\n[FULL BACKTEST] Filtering to backtest period BEFORE running...")
    final_params = wf_results["param_history"][-1] if wf_results["param_history"] else {}
    final_params = {k: v for k, v in final_params.items() if k != "fold"}
    detector = ThresholdRegimeDetectorWithIBS(config, **final_params)
    df_regimes_full = detector.detect_regimes(df_full)
    signals_full, diagnostics_full = detector.generate_signals_with_ibs(df_full)
    backtest_start = pd.to_datetime(config.start_date)
    backtest_end = pd.to_datetime(config.end_date)
    df_backtest = df_regimes_full[
        (df_regimes_full.index >= backtest_start) & (df_regimes_full.index <= backtest_end)
    ]
    signals_backtest = signals_full[
        (signals_full.index >= backtest_start) & (signals_full.index <= backtest_end)
    ]
    diagnostics_backtest = diagnostics_full[
        (diagnostics_full.index >= backtest_start) & (diagnostics_full.index <= backtest_end)
    ]
    if benchmark_returns is not None:
        benchmark_backtest = benchmark_returns[
            (benchmark_returns.index >= backtest_start) & (benchmark_returns.index <= backtest_end)
        ]
    else:
        benchmark_backtest = None
    print(f"  Backtest period: {df_backtest.index[0].date()} to {df_backtest.index[-1].date()}")
    print(f"  Trading days: {len(df_backtest)}")
    engine = BacktestEngine(config, benchmark_backtest)
    bt_results = engine.run_backtest(df_backtest, signals_backtest)
    final_value = bt_results["portfolio_value"].iloc[-1]
    total_return = (final_value / config.initial_capital - 1) * 100
    print(f"\n  Initial Capital: ${config.initial_capital:,.2f}")
    print(f"  Final Value: ${final_value:,.2f}")
    print(f"  Total Return: {total_return:.2f}%")
    print(f"  Trades: {bt_results['is_execution_day'].sum()}")
    print(
        f"  LONG: {(bt_results['position_type'] == 'LONG').sum()}, SHORT: {(bt_results['position_type'] == 'SHORT').sum()}, FLAT: {(bt_results['position_type'] == 'FLAT').sum()}"
    )
    print("\n[EXPORT]")
    out = f"results/ibs-volatility-thresholds/{TICKER.lower()}_v23"
    bt_results.to_csv(f"{out}_backtest.csv")
    pd.DataFrame(wf_results["fold_results"]).to_csv(f"{out}_walkforward.csv", index=False)
    pd.DataFrame(wf_results["param_history"]).to_csv(f"{out}_params.csv", index=False)
    diagnostics_backtest.to_csv(f"{out}_diagnostics.csv")
    with open(f"{out}_metrics.json", "w") as f:
        json.dump(wf_results["aggregate_metrics"], f, indent=2, default=str)
    print(f"  Saved to: {out}_*.csv/json")
    agg = wf_results["aggregate_metrics"]
    print(f"\n{'=' * 80}\nSUMMARY: OOS Sharpe={agg['oos_sharpe']:.3f}, GRI={agg['gri']:.3f}")
    print(f"DEPLOYMENT: {('READY' if agg['is_robust'] else 'NOT READY')}\n{'=' * 80}")
    return {
        "config": config,
        "backtest": bt_results,
        "wf": wf_results,
        "diagnostics": diagnostics_backtest,
    }


if __name__ == "__main__":
    results = main()
